In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import panel as pn
pn.extension()

import hvplot.pandas

In [14]:
df = pd.read_csv("defects_data.csv")

In [15]:
df

,defect_id,product_id,defect_type,defect_date,defect_location,severity,inspection_method,repair_cost
0,1,15,Structural,6/6/2024,Component,Minor,Visual Inspection,245.47
1,2,6,Functional,4/26/2024,Component,Minor,Visual Inspection,26.87
2,3,84,Structural,2/15/2024,Internal,Minor,Automated Testing,835.81
3,4,10,Functional,3/28/2024,Internal,Critical,Automated Testing,444.47
4,5,14,Cosmetic,4/26/2024,Component,Minor,Manual Testing,823.64
...,...,...,...,...,...,...,...,...
995,996,25,Structural,3/1/2024,Component,Minor,Automated Testing,813.14
996,997,23,Functional,3/21/2024,Component,Moderate,Automated Testing,944.07
997,998,17,Structural,1/16/2024,Component,Minor,Automated Testing,401.12
998,999,96,Cosmetic,6/21/2024,Internal,Moderate,Manual Testing,775.63


In [16]:
idf = df.interactive()

## 1) Variation in repair_cost over the months of 2024 by inspection_method

In [18]:
import panel as pn
import pandas as pd
import hvplot.pandas

pn.extension()

# Ensure datetime
df['defect_date'] = pd.to_datetime(df['defect_date'])

months = ["January","February","March","April","May","June",
          "July","August","September","October","November","December"]

month_slider = pn.widgets.DiscreteSlider(
    name="Year 2024",
    options=months,
    value="January"
)

def heatmap_plot(selected_month):
    filtered = df[df['defect_date'].dt.strftime('%B') == selected_month]

    heat_data = filtered.pivot_table(
        index="severity",
        columns="inspection_method",
        values="repair_cost",   # can be any column
        aggfunc="count"         # count occurrences
    ).fillna(0)

    return heat_data.hvplot.heatmap(
        title=f"Defect Count Heatmap ({selected_month} 2024)",
        cmap="greens",
        height=400,
        width=600
    )

heatmap_bound = pn.bind(heatmap_plot, month_slider)

pn.Column(month_slider, heatmap_bound).servable()

Column
    [0] DiscreteSlider(name='Year 2024', options=['January', 'February', ...], value='January')
    [1] ParamFunction(function, _pane=HoloViews, defer_load=False)

In [19]:
import panel as pn
import pandas as pd
import hvplot.pandas

pn.extension()

months = ["January","February","March","April","May","June",
          "July","August","September","October","November","December"]

month_slider = pn.widgets.DiscreteSlider(
    name="Year 2024",
    options=months,
    value="January"
    )

def plot_month(selected_month):
    # filter dataframe by month name
    filtered = df[df['defect_date'].dt.strftime('%B') == selected_month]

    monthly = filtered.groupby(['defect_date','inspection_method'])['repair_cost'].mean().reset_index()

    return monthly.hvplot.line(
        x='defect_date',
        y='repair_cost',
        by='inspection_method',
        legend='top',
        title=f"Repair Cost Trend",
        height = 400
    )

bound_plot = pn.bind(plot_month, month_slider)

pn.Column(month_slider, bound_plot).servable()

Column
    [0] DiscreteSlider(name='Year 2024', options=['January', 'February', ...], value='January')
    [1] ParamFunction(function, _pane=HoloViews, defer_load=False)

In [7]:
def pareto_percent(df):
    pareto = df['defect_type'].value_counts().reset_index()
    pareto.columns = ['defect_type', 'count']

    pareto['percent'] = 100 * pareto['count'] / pareto['count'].sum()
    pareto['cum_percent'] = pareto['percent'].cumsum()

    bars = pareto.hvplot.bar(
        x='defect_type',
        y='percent',
        title="Pareto Chart: Defect Type Percentage Contribution",
        ylabel="Percentage (%)",
        height=400,
        width=500
    )

    line = pareto.hvplot.line(
        x='defect_type',
        y='cum_percent',
        line_width=2
    )

    return bars * line

pareto_percent(df)

:Overlay
   .Bars.I  :Bars   [defect_type]   (percent)
   .Curve.I :Curve   [defect_type]   (cum_percent)

In [8]:
def pareto_percent(df):
    pareto = df['inspection_method'].value_counts().reset_index()
    pareto.columns = ['inspection_method', 'count']

    pareto['percent'] = 100 * pareto['count'] / pareto['count'].sum()
    pareto['cum_percent'] = pareto['percent'].cumsum()

    bars = pareto.hvplot.bar(
        x='inspection_method',
        y='percent',
        title="Pareto Chart: Inspection method Percentage Contribution",
        ylabel="Percentage (%)",
        height=400,
        width=500
    )

    line = pareto.hvplot.line(
        x='inspection_method',
        y='cum_percent',
        line_width=2
    )

    return bars * line

pareto_percent(df)

:Overlay
   .Bars.I  :Bars   [inspection_method]   (percent)
   .Curve.I :Curve   [inspection_method]   (cum_percent)

In [9]:
def pareto_percent(df):
    # Step 1: Filter out the "Cosmetic" defects to focus on Internal-Cost factors
    internal_df = df[df['defect_type'] != 'Cosmetic']

    # Step 2: Use the filtered dataframe for the Pareto analysis
    pareto = internal_df['inspection_method'].value_counts().reset_index()
    pareto.columns = ['inspection_method', 'count']

    pareto['percent'] = 100 * pareto['count'] / pareto['count'].sum()
    pareto['cum_percent'] = pareto['percent'].cumsum()

    bars = pareto.hvplot.bar(
        x='inspection_method',
        y='percent',
        title="Pareto Chart: Internal Inspection Methods (Excludes Cosmetics)",
        ylabel="Percentage (%)",
        height=400,
        width=500
    )

    line = pareto.hvplot.line(
        x='inspection_method',
        y='cum_percent',
        color='red', 
        line_width=2
    )

    return bars * line

pareto_percent(df)

:Overlay
   .Bars.I  :Bars   [inspection_method]   (percent)
   .Curve.I :Curve   [inspection_method]   (cum_percent)

In [10]:

import panel as pn
import pandas as pd
import hvplot.pandas  # enables df.hvplot

pn.extension()

# Widget for selecting column
y_axis_univariate_analysis = pn.widgets.RadioButtonGroup(
    name="Y axis",
    options=["defect_type", "defect_location", "severity"],
    button_type="success"
)

# Function to generate univariate plot
def univariate_plot(df, column):
    counts = df[column].value_counts().reset_index()
    counts.columns = [column, "count"]

    counts["percentage"] = (counts["count"] / len(df)) * 100

    return counts.hvplot.bar(
        x=column,
        y="percentage",
        title=f"{column} Distribution (%) over 2024",
        ylabel="Percentage (%)",
        xlabel=column,
        height=400,
        width=500
       
    )

# Make it interactive
interactive_univariate = pn.bind(univariate_plot, df=df, column=y_axis_univariate_analysis)

# Display dashboard
pn.Column(
    y_axis_univariate_analysis,
    interactive_univariate
).servable()

Column
    [0] RadioButtonGroup(button_type='success', name='Y axis', options=['defect_type', ...], value='defect_type')
    [1] ParamFunction(function, _pane=HoloViews, defer_load=False)

In [11]:
def pareto_summary_table(df):
    result = df.groupby('defect_type')['repair_cost'].sum().reset_index()
    result['percentage'] = (result['repair_cost'] / result['repair_cost'].sum()) * 100
    
    # Force two decimal places for the table view
    result = result.round(2) 

    return result.hvplot.table(
        columns=['defect_type', 'repair_cost', 'percentage'],
        width=500
    )

In [12]:
pareto_summary_table(df)

:Table   [defect_type,repair_cost,percentage]

## Creating a Dashboard

In [155]:
row_1 = pn.Row(
    pn.Column(heatmap_bound, sizing_mode="stretch_both"),
    pn.Column(bound_plot, sizing_mode="stretch_both"),
    sizing_mode="stretch_both",
    height=400  
)

row_2 = pn.Row(
    pn.Column(pareto_percent(df), sizing_mode="stretch_both"),
    pn.Column(y_axis_univariate_analysis, interactive_univariate, sizing_mode="stretch_both"),
    sizing_mode="stretch_both",
    height=400
)

template = pn.template.FastListTemplate(
    title='Manufacturing Defects Dashboard',
    main=[month_slider,
        pn.Column(
            row_1, 
            row_2, 
            sizing_mode="stretch_both"
        )
    ],
    accent_base_color="#88d8b0",
    header_background="#88d8b0",
    main_max_width="100%" 
)

template.show()

Launching server at http://localhost:2834
